# Clase 02 · Corpus, preparación y tokenización

23 de septiembre de 2026 · 15:00–17:00 · Versión 2026-09-23.

Continúa entrada, salida, corpus y etiquetas de Clase 1. Mantiene solicitudes.csv y casos_limpieza.json. El laboratorio evaluable 1 dura 40 min en clase y permite hasta 20 min adicionales; 2.5%, entrega hoy 23:59. La demostración guiada no sustituye tus decisiones ni resuelve Actividad 1.

## Entorno

**Colab:** sube este notebook, `solicitudes.csv` y `casos_limpieza.json` al panel Archivos. El código se ejecuta en una máquina remota. Descarga `resultados_clase02/` antes de cerrar; guardar el notebook no guarda automáticamente esos archivos. CPU suficiente.

Si las bibliotecas aún no están instaladas, ejecuta en una celda aparte:

```python
%pip install pandas nltk spacy
```

Después reinicia la sesión y ejecuta este notebook desde arriba. Si los imports ya funcionan, no necesitas reinstalar las bibliotecas. En Python 3.13 no copies el requisito `numpy==1.26.4` del perfil local 3.11/3.12; conserva las dependencias compatibles del entorno. El manifiesto registra las versiones utilizadas.


## Preparación · modelo de español

**spaCy** es la biblioteca; **es_core_news_sm** es un paquete separado con componentes entrenados para español. Tokenizar con `spacy.blank("es")` no requiere ese paquete, pero nuestra demostración de lematización sí.

La siguiente celda comprueba el modelo antes de comenzar la práctica. Si falta, lo descarga con la herramienta oficial de spaCy usando el mismo Python del notebook; requiere conexión a internet. Si ya existe, lo reutiliza. Luego carga el paquete directamente para evitar depender de la detección de paquetes recién instalados dentro de la sesión.

Si la descarga falla, revisa el error de instalación y la conexión. Si aparece una incompatibilidad después de instalar, reinicia la sesión y ejecuta desde arriba. Esta celda no reemplaza el modelo por resultados escritos a mano.


In [10]:
import importlib
import importlib.util
import subprocess
import sys
import spacy

modelo = "es_core_news_sm"
if importlib.util.find_spec(modelo) is None:
    print("Descargando el modelo de español compatible con spaCy...")
    subprocess.run(
        [sys.executable, "-m", "spacy", "download", modelo],
        check=True,
    )
    importlib.invalidate_caches()

# Importar el paquete directamente también funciona después de instalarlo.
paquete_es = importlib.import_module(modelo)
nlp = paquete_es.load()
print("Modelo listo:", modelo, nlp.meta["version"], "| spaCy:", spacy.__version__)


Modelo listo: es_core_news_sm 3.8.0 | spaCy: 3.8.16


In [11]:
from pathlib import Path
import sys, platform, json, re, unicodedata
import pandas as pd
from importlib.metadata import version
print("Python:", platform.python_version(), "pandas:", version("pandas"))
# Local: busca 07_datos en el curso. Colab: sube los archivos al panel Archivos.
candidates = [Path.cwd(), *Path.cwd().parents]
DATA = next((p / '07_datos' for p in candidates if (p / '07_datos/solicitudes.csv').exists()), Path.cwd())
if not (DATA / 'solicitudes.csv').exists():
    raise FileNotFoundError('Sube solicitudes.csv al panel Archivos de Colab, o abre el notebook dentro de curso_pln_tec.')
OUT = Path.cwd() / 'resultados_clase02'
OUT.mkdir(exist_ok=True)


Python: 3.12.10 pandas: 3.0.5


## Paso 1 · Importar y cargar

`pandas` permite inspeccionar tablas; `json` lee la lista de casos difíciles. `re` y `unicodedata` vienen con Python. Conserva ambos archivos de datos en `07_datos/` local o en el panel Archivos de Colab. Su origen es sintético, sin personas reales.

In [12]:
df = pd.read_csv(DATA / 'solicitudes.csv', encoding='utf-8')
cases = json.loads((DATA / 'casos_limpieza.json').read_text(encoding='utf-8'))
print('Filas:', len(df), 'familias:', df['family'].nunique())

Filas: 144 familias: 48


## Paso 2 · Inspeccionar antes de transformar

`head()` muestra cinco filas. `isna().sum()` cuenta ausencias por columna. Un texto vacío puede ser `""`, no solo un valor nulo: se comprueba aparte. Una etiqueta ausente no se inventa. En este corpus aprobado puede haber cero faltantes; no inventes problemas que el conteo no muestra.

In [13]:
print(df[['id','text','label','family','split']].head().to_string(index=False))
print('\nFaltantes:', df.isna().sum().to_dict())
print('Textos vacíos:', int(df['text'].fillna('').str.strip().eq('').sum()))
print('Etiquetas:', df['label'].value_counts().to_dict())

  id                                                                                    text  label    family split
S001                                          Olvidé la contraseña y necesito restablecerla. acceso acceso_00 train
S002           Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar? acceso acceso_00 train
S003 Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito. acceso acceso_00 train
S004                                         El código de verificación no llega a mi correo. acceso acceso_01 train
S005          Buen día, el código de verificación no llega a mi correo. ¿Me pueden orientar? acceso acceso_01 train

Faltantes: {'id': 0, 'family': 0, 'split': 0, 'label': 0, 'text': 0, 'origin': 0}
Textos vacíos: 0
Etiquetas: {'acceso': 36, 'pagos': 36, 'conectividad': 36, 'software': 36}


## Paso 3 · Duplicados y familias

`duplicated()` marca repeticiones exactas después de su primera aparición. No se llama `drop_duplicates()` ciegamente: textos iguales pueden pertenecer a eventos distintos o tener etiquetas contradictorias. `family` reúne variantes del mismo patrón; deben permanecer juntas al separar datos.

In [14]:
print('IDs repetidos:', int(df['id'].duplicated().sum()))
print('Textos repetidos:', int(df['text'].duplicated().sum()))
print('Conflictos por texto:', int((df.groupby('text')['label'].nunique() > 1).sum()))
print(df[df['family'] == df.iloc[0]['family']][['id','text','family','split']].to_string(index=False))

IDs repetidos: 0
Textos repetidos: 0
Conflictos por texto: 0
  id                                                                                    text    family split
S001                                          Olvidé la contraseña y necesito restablecerla. acceso_00 train
S002           Buen día, olvidé la contraseña y necesito restablecerla. ¿Me pueden orientar? acceso_00 train
S003 Olvidé la contraseña y necesito restablecerla; ya reinicié y sigue igual, urge tantito. acceso_00 train


## Paso 4 · Unicode no es quitar acentos

Un punto de código identifica una unidad de Unicode. Una letra visible puede estar compuesta por varios. NFC unifica representaciones canónicamente equivalentes: no convierte «sí» en «si». `len` cuenta puntos de código, no necesariamente letras visibles. Una codificación como UTF-8 convierte esos códigos en bytes para guardar el archivo.

In [15]:
a = 'á'
b = 'a\u0301'
print('Antes:', a == b, len(a), len(b))
print('Después de NFC:', unicodedata.normalize('NFC', b) == a)
print('UTF-8:', list(a.encode('utf-8')))

Antes: False 1 2
Después de NFC: True
UTF-8: [195, 161]


## Paso 5 · Crear una normalización conservadora

La función recibe un texto, unifica Unicode, reduce secuencias de espacios y recorta extremos. Conserva acentos, negación, signos y mayúsculas. `r"\s+"` significa una o más unidades de espacio. Su salida se añade a otra columna para no perder la fuente.

In [17]:
def normalize(texto):
    if not isinstance(texto, str):
        raise TypeError('Se esperaba texto; revisar faltantes antes de normalizar.')
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFC', texto)).strip()
clean = df.assign(text_normalized=df['text'].map(normalize))
print(repr(normalize('  No   quiero cancelar.  ')))
assert 'NO' in normalize('Quiero cancelar, NO renovar.')
assert normalize('contraseña') == 'contraseña'
assert clean['text'].equals(df['text'])

'No quiero cancelar.'


## Paso 6 · Comparar y detectar pérdidas

Este experimento compara NFC con un filtro ASCII deliberadamente destructivo. `string.printable` no incluye todas las letras del español. Lee cada antes/después y selecciona tres problemas verificables; no los confundas con tres faltantes inexistentes. La demostración da el mecanismo, tu entrega debe justificar la selección.

In [18]:
import string
comparacion = pd.DataFrame({'original': cases})
comparacion['conservador'] = comparacion['original'].map(normalize)
comparacion['ascii_destructivo'] = comparacion['original'].map(lambda t: ''.join(c for c in t if c in string.printable))
print(comparacion.to_string(index=False))

                                     original                                   conservador                             ascii_destructivo
                   No puedo entrar; sí pagué.                    No puedo entrar; sí pagué.                      No puedo entrar; s pagu.
                  Pésimo… ¡qué buen servicio!                   Pésimo… ¡qué buen servicio!                       Psimo qu buen servicio!
Mi correo ficticio es persona@example.invalid Mi correo ficticio es persona@example.invalid Mi correo ficticio es persona@example.invalid
                                  contraseña                                    contraseña                                    contrasena
                 Quiero cancelar, NO renovar.                  Quiero cancelar, NO renovar.                  Quiero cancelar, NO renovar.
           No carga el wifi, está bien gacho.            No carga el wifi, está bien gacho.             No carga el wifi, est bien gacho.
   No agarra el güifi; ya reinicie

## Paso 7 · Regex y privacidad

Una expresión regular describe un patrón de caracteres. El patrón didáctico busca una secuencia, @, dominio y punto; `re.sub` sustituye coincidencias. No garantiza reconocer todos los correos válidos ni anonimizar nombres o contexto. Las posiciones de caracteres cambian después de sustituir: conserva original y revisa anotaciones.

In [19]:
patron_correo = r'[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}'
texto = 'Escríbeme a persona@example.invalid'
print(re.findall(patron_correo, texto))
print(re.sub(patron_correo, '[CORREO]', texto))

['persona@example.invalid']
Escríbeme a [CORREO]


## Paso 8 · Tokenizar y separar oraciones · demostración guiada

spaCy es una biblioteca con reglas de tokenización y componentes lingüísticos. `blank("es")` crea un procesador de español sin modelo estadístico; `sentencizer` añade segmentación de oraciones por puntuación. Un token puede ser palabra, símbolo o cantidad. Separar por espacios perdería esta distinción. No añadimos nuevas exigencias a la rúbrica del laboratorio.

In [20]:
import spacy
nlp_reglas = spacy.blank('es')
nlp_reglas.add_pipe('sentencizer')
texto = 'No puedo pagar $1,250.50. Necesito ayuda.'
doc = nlp_reglas(texto)
print('Tokens:', [t.text for t in doc])
print('Oraciones:', [s.text for s in doc.sents])
print('Posiciones:', [(t.text, t.idx) for t in doc])

Tokens: ['No', 'puedo', 'pagar', '$', '1,250.50', '.', 'Necesito', 'ayuda', '.']
Oraciones: ['No puedo pagar $1,250.50.', 'Necesito ayuda.']
Posiciones: [('No', 0), ('puedo', 3), ('pagar', 9), ('$', 15), ('1,250.50', 16), ('.', 24), ('Necesito', 26), ('ayuda', 35), ('.', 40)]


## Paso 9 · Stemming y lematización · demostración guiada

NLTK reúne algoritmos para explorar lenguaje; aquí usamos Snowball en español, sin descargar colecciones. Recorta formas a una raíz aproximada. spaCy con `es_core_news_sm` añade análisis lingüístico para obtener un lema, forma de diccionario. Los errores son posibles; la política de normalización no aplica ninguna de las dos técnicas automáticamente. El modelo se comprobó y cargó en la preparación inicial como `nlp`. `corr` es una raíz aproximada, no un error: puede no ser una palabra. El lema se calcula con el modelo y el contexto; no presentes una lista escrita a mano como resultado.

In [22]:
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer('spanish')
print('Stems:', [(w, stemmer.stem(w)) for w in ['corriendo','corrió','correrá']])
# nlp se cargó en la preparación inicial.
frase = nlp('Ella está corriendo, ayer corrió y mañana correrá.')
print('Lemas:', [(t.text, t.lemma_) for t in frase if t.text in {'corriendo','corrió','correrá'}])
print('Versiones:', spacy.__version__, nlp.meta['version'], version('nltk'))

Stems: [('corriendo', 'corr'), ('corrió', 'corr'), ('correrá', 'corr')]
Lemas: [('corriendo', 'correr'), ('corrió', 'correr'), ('correrá', 'correr')]
Versiones: 3.8.16 3.8.0 3.10.3


## Paso 10 · Conservar y verificar las particiones

Train servirá para aprender; validation para escoger decisiones; test para la evaluación final. El corpus ya tiene estas particiones por familia: no se sortean filas de nuevo. `nunique` cuenta valores distintos: una familia debe tener solo un nombre de partición. Cero solapamientos ayuda a impedir que variantes del mismo caso aparenten generalización.

In [23]:
def validate_splits(tabla):
    required = {'id', 'text', 'label', 'family', 'split'}
    if not required <= set(tabla):
        raise ValueError('Faltan columnas del contrato.')
    if tabla['id'].duplicated().any() or tabla['text'].isna().any():
        raise ValueError('IDs duplicados o textos ausentes.')
    if (tabla.groupby('family')['split'].nunique() > 1).any():
        raise ValueError('Fuga: familia compartida entre particiones.')
    if (tabla.groupby('text')['split'].nunique() > 1).any():
        raise ValueError('Fuga: texto idéntico en distintas particiones.')
    if set(tabla['split']) != {'train', 'validation', 'test'}:
        raise ValueError('Se requieren las tres particiones.')
    return True
assert validate_splits(clean)
print(clean.groupby('split').agg(filas=('id','size'), familias=('family','nunique')))
particiones = {nombre: clean[clean['split'] == nombre].copy() for nombre in ['train','validation','test']}
print('Se conservaron los IDs y particiones originales.')

            filas  familias
split                      
test           24         8
train          96        32
validation     24         8
Se conservaron los IDs y particiones originales.


## Paso 11 · Reto original: contaminación controlada

En una copia, mueve una variante de una familia a otra partición y demuestra que el control detecta el solapamiento. No alteres `df` ni los datos fuente. Predice el resultado, captura el error y explica por qué un reparto por filas sería débil. Completa aquí tu evidencia y hasta 150 palabras; el enunciado de Laboratorio 1 mantiene su rúbrica.

In [24]:
bad = df.copy()

# Elegimos una familia del split 'train' y se mueve solo una fila
familia_objetivo = bad.loc[bad['split'] == 'train', 'family'].iloc[0]
fila_a_mover = bad[bad['family'] == familia_objetivo].index[0]
bad.loc[fila_a_mover, 'split'] = 'test'

print("Familia elegida        :", familia_objetivo)
print("Fila movida (id)       :", bad.loc[fila_a_mover, 'id'])
print("Splits de esa familia  :",
      bad[bad['family'] == familia_objetivo]['split'].value_counts().to_dict())

# Comprobación
try:
    validate_splits(bad)
    print("ERROR: el control NO detectó la contaminación.")
except ValueError as e:
    print("ValueError capturado   :", e)

# Verificación de que df original sigue intacto
assert validate_splits(df)  # df sigue siendo válido
print("df original intacto: validate_splits(df) == True")

# Evidencia 
evidencia = (
    "Predije que validate_splits detectaría la fuga porque la familia "
    f"'{familia_objetivo}' quedaría repartida entre 'train' y 'test'. "
    "Al mover una sola fila, groupby('family')['split'].nunique() devuelve 2 "
    "y la comprobación lanza ValueError('Fuga: familia compartida entre "
    "particiones.'). Esto demuestra por qué un reparto por filas sería débil: "
    "variantes casi idénticas del mismo patrón caerían en train y test a la vez, "
    "y el modelo parecería generalizar cuando en realidad memorizó la familia. "
    "El control por familia evita esa ilusión; mover la familia completa, en "
    "cambio, no elimina todo riesgo, porque pueden existir otras dependencias "
    "(usuario, periodo, plantilla) que también habría que auditar. df original "
    "permanece válido."
)
print("\nEvidencia:\n", evidencia)

Familia elegida        : acceso_00
Fila movida (id)       : S001
Splits de esa familia  : {'train': 2, 'test': 1}
ValueError capturado   : Fuga: familia compartida entre particiones.
df original intacto: validate_splits(df) == True

Evidencia:
 Predije que validate_splits detectaría la fuga porque la familia 'acceso_00' quedaría repartida entre 'train' y 'test'. Al mover una sola fila, groupby('family')['split'].nunique() devuelve 2 y la comprobación lanza ValueError('Fuga: familia compartida entre particiones.'). Esto demuestra por qué un reparto por filas sería débil: variantes casi idénticas del mismo patrón caerían en train y test a la vez, y el modelo parecería generalizar cuando en realidad memorizó la familia. El control por familia evita esa ilusión; mover la familia completa, en cambio, no elimina todo riesgo, porque pueden existir otras dependencias (usuario, periodo, plantilla) que también habría que auditar. df original permanece válido.


## Paso 12 · Guardar el corpus y las particiones

CSV conserva columnas e IDs, no funciones de Python. `index=False` evita una columna extra. El manifiesto registra versiones, política y huella SHA-256: una huella permite detectar cambios del archivo, no demuestra que el contenido sea correcto. Reabrimos el CSV para comprobarlo.

In [25]:
import hashlib
clean.to_csv(OUT / 'corpus_validado.csv', index=False, encoding='utf-8')
for nombre, tabla in particiones.items():
    tabla.to_csv(OUT / f'{nombre}.csv', index=False, encoding='utf-8')
recargado = pd.read_csv(OUT / 'corpus_validado.csv', encoding='utf-8')
assert recargado['id'].tolist() == clean['id'].tolist()
assert validate_splits(recargado)
manifiesto = {'python': platform.python_version(), 'pandas': pd.__version__,
              'spacy': spacy.__version__, 'nltk': version('nltk'),
              'modelo_es': modelo, 'modelo_es_version': nlp.meta['version'],
              'politica': 'NFC y espacios; conservar original, negación, acentos y split',
              'sha256': hashlib.sha256((OUT/'corpus_validado.csv').read_bytes()).hexdigest()}
(OUT / 'manifiesto.json').write_text(json.dumps(manifiesto, ensure_ascii=False, indent=2), encoding='utf-8')
print(manifiesto)

{'python': '3.12.10', 'pandas': '3.0.5', 'spacy': '3.8.16', 'nltk': '3.10.3', 'modelo_es': 'es_core_news_sm', 'modelo_es_version': '3.8.0', 'politica': 'NFC y espacios; conservar original, negación, acentos y split', 'sha256': '724c86d3ae3cf7fd4d1e1fd3cbadc2f6b4bf036b762e1fd78a54365ac50e6357'}


## Paso 13 · Interpretación y Actividad 1

Explica tres problemas de calidad, una regla probada y la comprobación de familias. La entrega del laboratorio conserva el notebook y hasta 150 palabras, sin nuevo informe largo.

Actividad 1 (8%): asignada 23/09, entrega 29/09 a las 23:59. Usa su corpus y enunciado propios; requiere auditoría, original y procesado, diccionario, decisiones, particiones y reconstrucción. No requiere entrenar un modelo. Esta clase enseña los mecanismos; no entrega la solución completa de ese corpus.

Comprobación: ¿borrar «no» de «No quiero cancelar» simplifica sin costo? No: cambia la intención; la conservación se prueba con casos difíciles. ¿Mover una familia completa elimina todo riesgo? No: pueden existir otras dependencias, como usuario o periodo, que hay que auditar.

## Referencias

Python Unicode: https://docs.python.org/3/library/unicodedata.html

spaCy: https://spacy.io/usage/linguistic-features

NLTK Snowball: https://www.nltk.org/api/nltk.stem.snowball.html

Particiones y fuga: https://scikit-learn.org/stable/common_pitfalls.html

Datos sintéticos originales del curso. No se redistribuyen materiales comerciales.